In [17]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

In [18]:
load_dotenv()

True

In [24]:
#create a llm instance 
llm=ChatGoogleGenerativeAI(
        model='gemini-3.8-flash'
    )

In [20]:
#define state
class llmState(TypedDict):
    question: str
    answer: str

In [21]:
#define the node llm_qa
def llm_qa(state: llmState)->llmState:
    question=state['question']

    #create a prompt to feed to llm
    prompt=f'Answer the following question: \n {question}'

    #ask question from LLM
    answer=llm.invoke(prompt)

    #update the answer in  state
    state['answer']=answer

    return state

In [22]:
# create node in the graph
graph=StateGraph(llmState)

graph.add_node('llm_qa', llm_qa)

#define edges in the graph
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

workflow=graph.compile()

In [31]:
#invoke the graph 
initial_state={'question': 'what are the main components of Langgraph'}
final_state=workflow.invoke(initial_state)
answer=final_state['answer'].content
print(answer[0]['text'])

**LangGraph** is an orchestration framework by LangChain designed for building robust, stateful, and multi-actor applications (like AI agents) using Large Language Models. It models workflows as **graphs**.

The main components of LangGraph can be broken down into **Core Architecture Components** and **Advanced Control Components**:

---

### 1. Core Architecture Components

#### A. State
The **State** is the central data structure that represents the current snapshot of your application.
* **How it works:** It acts as the shared memory accessible by every part of the graph. When a component runs, it reads the current state and returns an update.
* **Implementation:** Usually defined using a Python `TypedDict` or a Pydantic `BaseModel`.
* **Reducers:** Attributes in the state can have "reducers" (e.g., `operator.add`) which specify *how* updates are applied (e.g., appending new messages to a list rather than overwriting the entire history).

#### B. Nodes
**Nodes** are the workers of t